# Baseline 1: model-only inference

Scores the unmodified Qwen3.5-0.8B on MATH with one greedy attempt per problem and no skills. It evaluates the 1,000-problem dev set and then MATH-500.

Setup, needed once per account:

1. Select a GPU in **Runtime > Change runtime type**. An A100 or L4 is much faster than a T4.
2. Add a read-only GitHub token for this repository in Colab's **Secrets** panel (the key icon) as `GH_TOKEN`, and turn on notebook access.

Then choose **Runtime > Run all** and check every box on the Google Drive permission popup. Keep this tab open while it runs. If the session disconnects, choose **Run all** again and it continues from where it stopped.

In [ ]:
# Settings. Change these before running if needed.
USE_GOOGLE_DRIVE = True  # False keeps results on the runtime's disk, erased when the session ends.
RUN_NAME = "pre-rl"
EVAL_BATCH_SIZE = 32
RUN_PUBLISHED_SCORE_CHECK = True  # Runs last, so a failure cannot affect the main results.

In [ ]:
import json
import os
import shutil
import subprocess
from pathlib import Path

from google.colab import drive, userdata

if shutil.which("nvidia-smi") is None:
    raise RuntimeError("No GPU found. Select one in Runtime > Change runtime type, then run all.")
get_ipython().system("nvidia-smi --query-gpu=name,memory.total --format=csv")

if USE_GOOGLE_DRIVE:
    drive.mount("/content/drive")
    RUN_ROOT = Path("/content/drive/MyDrive/self-derived-skills/runs")
else:
    RUN_ROOT = Path("/content/runs")
RUN_DIR = RUN_ROOT / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)
print(f"Results will be saved to {RUN_DIR}")

REPO_URL = "https://github.com/Ruichu-Xia/self-derived-skills.git"
REPO_DIR = "/content/self-derived-skills"

# The token stays in an environment variable so that it never appears in a command, URL, or error.
GIT_ENV = {**os.environ, "GH_TOKEN": userdata.get("GH_TOKEN")}
CREDENTIAL_HELPER = (
    "credential.helper=!f() { echo username=x-access-token; echo password=$GH_TOKEN; }; f"
)


def git(*args, cwd=None):
    """Run a git command that authenticates with the token in the environment."""
    subprocess.run(["git", "-c", CREDENTIAL_HELPER, *args], env=GIT_ENV, cwd=cwd, check=True)


if os.path.isdir(REPO_DIR):
    git("pull", "--ff-only", cwd=REPO_DIR)
else:
    git("clone", REPO_URL, REPO_DIR)
os.chdir(REPO_DIR)
get_ipython().system("git log -1 --format='%h %s'")

print("Installing packages. This takes a few minutes.")
install = subprocess.run(["bash", "scripts/colab_install.sh"], capture_output=True, text=True)
print(install.stdout[-1500:])
if install.returncode != 0:
    print(install.stderr[-4000:])
    raise RuntimeError("Installing the packages failed. See the output above.")

In [ ]:
def evaluate(split, name, adapter=None):
    """Run one evaluation, skipping finished work, and stop the notebook if it fails."""
    output_dir = RUN_DIR / name
    adapter_flag = f'--adapter "{adapter}"' if adapter else ""
    get_ipython().system(
        f"python experiments/evaluate_checkpoint.py --config configs/grpo_math.yaml "
        f'--output-dir "{output_dir}" --split {split} --batch-size {EVAL_BATCH_SIZE} {adapter_flag}'
    )
    metrics_path = output_dir / "metrics.json"
    if not metrics_path.exists():
        raise RuntimeError(f"Evaluation {name} failed. See the traceback above.")
    return json.loads(metrics_path.read_text())

## Evaluate the untrained model

In [ ]:
results = {
    "MATH dev (1,000 problems)": evaluate("dev", "dev"),
    "MATH-500": evaluate("math500", "math500"),
}

print(f"{'split':<28} {'n':>5} {'accuracy':>9} {'std err':>8} {'truncated':>10}")
for split, metrics in results.items():
    print(
        f"{split:<28} {metrics['n']:>5} {metrics['accuracy']:>9.3f} "
        f"{metrics['standard_error']:>8.3f} {metrics['truncated_frac']:>10.3f}"
    )

print(json.dumps(results, indent=2, sort_keys=True))

## Optional: reproduce the published score

MobileMoE reports 19.6% on MATH-500 for this model with lm-eval's 4-shot prompt and no chat template. This pipeline check uses a different prompt from the results above, so it is not a result to compare against. It runs in a separate environment because lm-eval needs an older parser than our reward.

In [ ]:
if RUN_PUBLISHED_SCORE_CHECK:
    check_dir = RUN_DIR / "published-score-check"
    get_ipython().system("pip install -q uv==0.13.7")
    get_ipython().system("uv venv -q --allow-existing --system-site-packages /content/lm_eval_env")
    get_ipython().system(
        'uv pip install -q --python /content/lm_eval_env/bin/python "lm-eval[math]==0.4.13"'
    )
    get_ipython().system(
        "/content/lm_eval_env/bin/python -m lm_eval --model hf "
        '--model_args "pretrained=Qwen/Qwen3.5-0.8B,'
        'revision=2fc06364715b967f1860aea9cf38778875588b17,dtype=float32" '
        f'--tasks minerva_math500 --batch_size 16 --device cuda --output_path "{check_dir}"'
    )